# TP3 (a completer) : Regression lineaire — *California Housing*

Remplacez chaque `...` et chaque `# TODO`. Corrige :
`../notebooks/TP3_regression_lineaire.ipynb`.

**Objectif.** Predire la **valeur mediane des logements** de 20 640 districts et
**interpreter** l'effet de chaque variable.

In [ ]:
# Cellule fournie : a executer telle quelle.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NAVY, ACCENT, GRAY = "#16284D", "#0EA5E9", "#5B6679"
RED = "#C0504D"
PALETTE = [ACCENT, NAVY, "#F79646", "#3FA45B", RED]
plt.rcParams.update({
    "figure.figsize": (7, 4.5), "font.size": 12,
    "axes.titlecolor": NAVY, "axes.titleweight": "bold",
    "axes.edgecolor": GRAY, "axes.spines.top": False, "axes.spines.right": False,
})
pd.set_option("display.width", 120)
print("Environnement pret.")

## Etape 0 : charger les donnees (fournie)

In [ ]:
from sklearn.datasets import fetch_california_housing
ds = fetch_california_housing(as_frame=True)
X, y = ds.data, ds.target.rename("prix")   # prix en x100 000 USD
X.head()

## 1. Exploration
**Consigne.** Affichez la correlation de chaque variable avec `prix`, triee.

In [ ]:
# TODO : correlations avec le prix (indice : pd.concat([X, y], axis=1).corr())
correlations = pd.concat([X, y], axis=1).corr()['prix'].sort_values(ascending=False)
print("Corrélations avec le prix :")
print(correlations)

## 2. Modelisation
**Consigne.** Split train/test (20% test, `random_state=42`), puis entrainez une
`LinearRegression`. Affichez l'ordonnee a l'origine et les coefficients.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
modele = LinearRegression()
modele.fit(X_train, y_train)    # TODO : entrainer la regression
# TODO : afficher intercept_ et coef_
print(f"Ordonnée à l'origine (intercept) : {modele.intercept_:.4f}")
print("\nCoefficients des variables :")
coefs = pd.Series(modele.coef_, index=X.columns)
print(coefs.round(4))

## 3. Evaluation
**Consigne.** Calculez **R2**, **RMSE** et **MAE** sur le test.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = modele.predict(X_test)
# TODO : R2, RMSE (= racine de mean_squared_error), MAE
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"R² : {r2:.4f}")
print(f"RMSE : {rmse:.4f} (x100 000 USD)")
print(f"MAE : {mae:.4f} (x100 000 USD)")

print(f"\nEn dollars :")
print(f"RMSE : ${rmse * 100000:.0f}")
print(f"MAE : ${mae * 100000:.0f}")

In [ ]:
 R², RMSE, MAE et interprétation
print("\n" + "="*60)
print("RÉPONSE QUESTION 1 : R², RMSE, MAE et interprétation")
print("="*60)
print(f"""
R²  : {r2:.4f} 
RMSE : {rmse:.4f} (x100 000 USD) soit environ {rmse*100000:.0f} $
MAE  : {mae:.4f} (x100 000 USD) soit environ {mae*100000:.0f} $

Interprétation :
 Le R² de {r2:.4f} signifie que {r2*100:.1f}% de la variance du prix est expliquée par le modèle.
 C'est un score correct pour des données socio-économiques complexes.
RMSE : en moyenne, l'erreur de prédiction est de ~{rmse*100000:.0f} $.
 MAE : en moyenne, l'erreur absolue est de ~{mae*100000:.0f} $.
 Le RMSE > MAE indique que le modèle fait quelques grosses erreurs.
 Le modèle est utile mais imparfait pour prédire les prix individuels.
""")
print("="*60)

## 4. Visualisation
**Consigne.** (a) Tracez `prix` vs `MedInc` avec la droite de regression simple.
(b) Tracez **predit vs reel** sur un echantillon. (c) Tracez les **residus**.

In [ ]:
# (a) prix vs MedInc + droite
simple = LinearRegression().fit(X[["MedInc"]], y)
xs = np.linspace(0, X["MedInc"].quantile(0.99), 100).reshape(-1, 1)
fig, ax = plt.subplots(figsize=(6.2, 4.6))
ax.scatter(X["MedInc"], y, s=8, color=ACCENT, alpha=0.2, edgecolor="none")
# TODO : tracer la droite simple.predict(xs)
y_simple = simple.predict(xs)
ax.plot(xs, y_simple, color=NAVY, linewidth=2, label='Régression simple')

ax.set(title="Prix vs revenu median", xlabel="MedInc", ylabel="prix (x100k USD)")
ax.legend()
plt.show()

In [ ]:
# (b) predit vs reel (echantillon de 2000 points)
rng = np.random.default_rng(0)
idx = rng.choice(len(y_test), 2000, replace=False)
yt, yp = y_test.to_numpy()[idx], y_pred[idx]
fig, ax = plt.subplots(figsize=(5.4, 5))
# TODO : scatter(yt, yp) + diagonale de reference
ax.scatter(yt, yp, s=10, color=ACCENT, alpha=0.4, edgecolor="none")
min_val = min(yt.min(), yp.min())
max_val = max(yt.max(), yp.max())
ax.plot([min_val, max_val], [min_val, max_val], color=NAVY, linestyle='--', linewidth=2, label='Prédiction parfaite')

ax.set(title="Prédit vs Réel", xlabel="Prix réel (x100k USD)", ylabel="Prix prédit (x100k USD)")
ax.legend()
plt.show()

In [ ]:
# (c) residus
residus = yt - yp
fig, ax = plt.subplots(figsize=(6.2, 4.4))
ax.axhline(0, color=NAVY, lw=2)
# TODO : scatter(yp, residus)
ax.scatter(yp, residus, s=10, color=ACCENT, alpha=0.4, edgecolor="none")

ax.set(title="Résidus", xlabel="Prix prédit (x100k USD)", ylabel="Résidu (x100k USD)")
ax.axhline(0, color='red', linestyle='--', alpha=0.5, label='Résidu = 0')
ax.legend()
plt.show()

In [ ]:
Ce que révèle le graphique des résidus
print("\n" + "="*60)
print("RÉPONSE QUESTION 2 : Interprétation du graphique des résidus")
print("="*60)
print("""
Le graphique des résidus révèle plusieurs choses importantes :

UN EFFET DE PLAFOND (effet de plafonnement) :
 Pour les prix prédits > 4.5 (soit > 450 000 $), les résidus sont négatifs
 Le modèle sous-estime systématiquement les prix élevés
 Le modèle ne peut pas prédire des prix > 5.0 (500 000 $)
 Cela suggère un plafond naturel dans les données ou une non-linéarité
 HÉTÉROSCÉDASTICITÉ (variance non constante) :
 La variance des résidus AUGMENTE avec le prix prédit
   Pour les prix bas (< 1.5) : résidus concentrés près de 0
   Pour les prix élevés (> 3.0) : résidus plus dispersés
   Les prédictions sont moins précises pour les prix élevés

ABSENCE DE BIAIS SYSTÉMATIQUE :
   Les résidus sont centrés autour de 0
   Pas de tendance linéaire évidente
   Le modèle n'est pas biaisé en moyenne
 CONCLUSION : Le modèle linéaire atteint ses limites pour les prix élevés.
Un modèle non-linéaire (avec transformation log ou polynôme) serait plus approprié.
""")
print("="*60)

## 5. Interpretation
**Consigne.** Affichez l'effet (en k USD) d'une unite supplementaire de revenu
median et d'un an d'age moyen du bati. Estimez le prix de 3 districts.

In [ ]:
coefs = pd.Series(modele.coef_, index=X.columns)
# TODO : afficher coefs['MedInc']*100 et coefs['HouseAge']*100
print("Effet d'une unité supplémentaire (en milliers de dollars) :")
print(f"Revenu médian (MedInc) : +${coefs['MedInc'] * 100000:.0f}")
print(f"Âge moyen du bâti (HouseAge) : +${coefs['HouseAge'] * 100000:.0f}")

estim = X.head(3).copy()
estim["prix_estime"] = modele.predict(X.head(3))  # TODO : modele.predict(X.head(3))
print("\nEstimation des prix pour 3 districts :")
print(estim[['MedInc', 'HouseAge', 'AveRooms', 'prix_estime']])
estim["prix_estime_dollars"] = estim["prix_estime"] * 100000
print("\nEstimation en dollars :")
print(estim[['MedInc', 'prix_estime', 'prix_estime_dollars']])

In [ ]:
Effet du revenu médian sur le prix
print("\n" + "="*60)
print("RÉPONSE QUESTION 3 : Effet du revenu médian sur le prix")
print("="*60)
print(f"""
L'effet du revenu médian (MedInc) sur le prix des logements :

Une unité supplémentaire de MedInc (c'est-à-dire +10 000 $ de revenu médian)
augmente le prix de : {coefs['MedInc'] * 100000:.0f} $ en moyenne.

INTERPRÉTATION :
 C'est l'effet le PLUS IMPORTANT de toutes les variables
 MedInc est la variable la plus prédictive (corrélation de 0.69)
 Augmenter le revenu médian de 10 000 $ dans un district
  augmente la valeur médiane des logements d'environ {coefs['MedInc'] * 100000:.0f} $.

EN TERMES DE POURCENTAGE :
 Le prix médian moyen est d'environ 2.08 (208 000 $)
 Une augmentation de 1 unité de MedInc (+10 000 $)
  représente une augmentation de {coefs['MedInc'] / y.mean() * 100:.1f}% du prix
  COMPARAISON AVEC LA CORRÉLATION :
Corrélation MedInc-prix : 0.69 (très forte)
Coefficient de régression : {coefs['MedInc']:.4f}
Cela confirme que MedInc est le principal déterminant du prix

PRUDENCE :
Cet effet est une corrélation, pas nécessairement un effet causal
 D'autres facteurs confondants peuvent exister
Le prix élevé des logements attire les ménages aisés (effet inverse possible)
""")
print("="*60)



In [ ]:
BONUS : Comparaison R² 
print("\n" + "="*60)
print("BONUS : Comparaison R²")
print("="*60)

simple = LinearRegression()
simple.fit(X_train[["MedInc"]], y_train)
y_pred_simple = simple.predict(X_test[["MedInc"]])
r2_simple = r2_score(y_test, y_pred_simple)

r2_all = r2_score(y_test, y_pred)

print(f"R² avec une seule variable (MedInc) : {r2_simple:.4f}")
print(f"R² avec toutes les variables : {r2_all:.4f}")
print(f"Amélioration : {r2_all - r2_simple:.4f}")

print("\nInterprétation :")
print(f"MedInc seul explique {r2_simple*100:.1f}% de la variance du prix.")
print(f"Les 8 variables ensemble expliquent {r2_all*100:.1f}% de la variance.")
print(f"Les autres variables apportent { (r2_all - r2_simple)*100:.1f}% d'information supplémentaire.")
print("Cela montre que MedInc est la variable la plus importante, mais que les autres")
print("variables (AveRooms, HouseAge, etc.) améliorent significativement la prédiction.")
print(f"""
R² avec MedInc seul : {r2_simple:.4f}
R² avec toutes les variables : {r2_all:.4f}
Amélioration : {(r2_all - r2_simple)*100:.1f}%

Conclusion : MedInc est la variable la plus importante (explique ~47% de la variance),
mais les 7 autres variables apportent ~13% d'information supplémentaire.
Le modèle complet est donc plus performant, mais MedInc reste le principal
déterminant du prix des logements en Californie.
""")

## A rendre
- R2, RMSE, MAE et leur interpretation.
- Ce que revele le graphique des residus (indice : effet de plafond).
- L'effet du revenu median sur le prix.

**Bonus.** Comparez le R2 avec une seule variable (`MedInc`) vs toutes.